<a href="https://colab.research.google.com/github/Jorge-Ruiz-Troccoli/Data-Science-II/blob/main/Clase%20014/ejemplo_1_clase_GridSearchCV_RandomizedSearchCV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mejora de modelos de ML parte II
### Coderhouse - Data Science
Profe Jorge Ruiz

Estaremos trabajando en el dataset: **Breast Cancer Wisconsin**

In [ ]:
#Importamos librerias y el Dataset
import pandas as pd
import numpy as np
import scipy as sp

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV

from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

In [ ]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
data

In [ ]:
#Convertimos en dataframe
df = pd.DataFrame(np.c_[data['data'], data['target']],
                  columns= np.append(data['feature_names'], ['target']))

In [ ]:
df.shape

In [ ]:
#Visualizamos el objeto
df.head()

In [ ]:
#Como son muchos atributos nos vamos a quedar unicamente con algunos de ellos
features= list(df.columns[0:10])
features

In [ ]:
#A lo que ya tenemos le agregamos la variable: target
data = df[features + ['target']]
data.head()

In [ ]:
#Separamos en X e y como así también en Train y Test
X = data.drop(['target'],axis=1)
y = data['target']

# Dividimos los datos en Train y Test
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

In [ ]:
round(y.value_counts(normalize=True)*100,1)

In [ ]:
train_test_split?

In [ ]:
X_train.shape, X_test.shape

In [ ]:
#Creamos nuestro objeto RF
clf =  RandomForestClassifier()

# GridSearch CV

In [ ]:
#Definicion de Hyperparámetros
param_grid = {'n_estimators': [100, 200],  # Número de árboles en el bosque
    'max_depth': [2,4,6],  # Profundidad máxima de cada árbol
    #'min_samples_split': [2,4,6],  # Número mínimo de muestras requeridas para dividir un nodo interno
    #'min_samples_leaf': [2, 4, 6],  # Número mínimo de muestras requeridas en cada hoja del árbol
    #'bootstrap': [True, False]  # Indica si se debe realizar bootstrap para construir los árboles
}

#Utilizamos la grilla definida anteriormente...
model_RF = GridSearchCV(clf, cv=5, param_grid=param_grid, scoring='f1', n_jobs=-1, verbose=3)

In [ ]:
%%time
#Entrenamos nuestro modelo de RF con la grilla ya definida y CV con tamaño de Fold=5
model_RF.fit(X_train, y_train)

In [ ]:
model_RF.get_params()

Entonces ... ¿Cómo sabemos cuales son los mejores hyperparámetros? Para ello, tendremos que analizar las siguientes funciones:
 * best_params_
 * best_score_
 * cv_results_

Aclaración: Se recomienda profundizar en la documentación asociada.

Link de Interes:
* https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

In [ ]:
print("Mejores parametros: "+str(model_RF.best_params_))
print("Mejor Score: "+str(model_RF.best_score_)+'\n')

In [ ]:
#Veamos los resultados obtenidos
scores = pd.DataFrame(model_RF.cv_results_)
scores.sort_values("rank_test_score")

In [ ]:
#Ahora sí ya estamos en condición de realizar nuestras predicciones
prediction = model_RF.predict(X_test)

In [ ]:
#Accuracy
print('Exactitud:', accuracy_score(y_test, prediction))

In [ ]:
# Matriz de Confusion
cm = confusion_matrix(y_test,prediction)
print("Matriz de confusión:")
print(cm)

# Halving Grid Search

In [ ]:
# se recomienda leer sobre https://es.wikipedia.org/wiki/Algoritmo_divide_y_vencer%C3%A1s

#Definicion de Hiperparámetros


param_grid = {'n_estimators': [400,500,600],  # Número de árboles en el bosque
    'max_depth': [18, 20,22],  # Profundidad máxima de cada árbol
    'min_samples_split': [5, 15],  # Número mínimo de muestras requeridas para dividir un nodo interno
    'min_samples_leaf': [4, 6]  # Número mínimo de muestras requeridas en cada hoja del árbol
    #'bootstrap': [True, False]  # Indica si se debe realizar bootstrap para construir los árboles
}
#Aplicamos la grilla al modelo
#Utilizamos la grilla definida anteriormente...
RF_halving_cv = HalvingGridSearchCV(clf, param_grid=param_grid, factor=3, min_resources=40, cv=5,  scoring='f1', n_jobs=-1, verbose=2)


In [ ]:
len(X_train)

In [ ]:
%%time
#Entrenamos RF con la grilla definida arriba y CV con tamaño de Fold=5
RF_halving_cv.fit(X_train, y_train)

In [ ]:
print("Mejores parametros", RF_halving_cv.best_params_)
print("Mejor CV score", RF_halving_cv.best_score_)
print(f'Accuracy del modelo = {round(accuracy_score(y_test, RF_halving_cv.predict(X_test)), 5)}')

# Random Search

In [ ]:
# https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.HalvingRandomSearchCV.html

In [ ]:
# Grilla para Random Search
#Definicion de Hyperparámetros
param_grid = {'n_estimators': [100, 200, 300,400,500],  # Número de árboles en el bosque
    'max_depth': [5, 10, 15,20,25],  # Profundidad máxima de cada árbol
    'min_samples_split': [1, 5, 10, 15,20],  # Número mínimo de muestras requeridas para dividir un nodo interno
    'min_samples_leaf': [4, 6, 8, 10],  # Número mínimo de muestras requeridas en cada hoja del árbol
    'bootstrap': [True, False]  # Indica si se debe realizar bootstrap para construir los árboles
}
#Aplicamos la grilla al modelo

model_RF = RandomizedSearchCV(clf, param_grid, cv=5,  scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=2)

In [ ]:
%%time
#Entrenamos KNN con la grilla definida arriba y CV con tamaño de Fold=5
model_RF.fit(X_train, y_train)

In [ ]:
print("Mejores parametros: "+str(model_RF.best_params_))
print("Mejor Score: "+str(model_RF.best_score_)+'\n')

In [ ]:
#Analizamos qué obtuvimos
scores = pd.DataFrame(model_RF.cv_results_)
scores

In [ ]:
#Prediccion
prediction = model_RF.predict(X_test)

In [ ]:
#Accuracy
print('Exactitud:', accuracy_score(y_test, prediction))

In [ ]:
# Matriz de Confusion
cm = confusion_matrix(y_test,prediction)
print("Matriz de confusión:")
print(cm)